# Experiment 7 — Matched robustness: Trad-style raw pixels vs decode-then-lexical

Uses exactly the 1,777 hostname-disjoint holdout samples from Experiment 6.
Both pipelines receive the SAME distorted, rendered QR for each sample.
- Visual: rendered QR -> fixed canvas/crop/downsample/binarize -> 4,761 pixels -> frozen Trad-style model.
- Lexical: rendered QR -> OpenCV decode -> exact deployment URL features -> frozen 10-feature model.

IMPORTANT: a decoder failure is an abstention for the lexical pipeline. A
nonempty but WRONG decoded payload counts as end-to-end failure, even if a
classifier gives it the correct label by coincidence. Correct decoding is only
knowable in an experiment with ground-truth payloads.

We provide a fixed, deliberately naive image preprocessing strategy for the
raw-pixel pipeline. Under rotation it is not rotation-invariant. Do NOT infer
robustness rankings for every possible visual frontend from this experiment.

The experiment REUSES the Experiment 4 decoder results if the validated
`sample_level_robustness_results.csv` is available; otherwise it re-decodes
directly. This saves time without changing the source QR or distortion code.

Only open .pickle/.pkl files from sources you trust.


## 1. Configuration — usually no edits required


In [ ]:
from pathlib import Path
import os, sys, json, pickle, time, zipfile, io, warnings
from concurrent.futures import ThreadPoolExecutor
import threading
import re
from urllib.parse import urlparse, urlunparse, urlsplit
import numpy as np
import pandas as pd
import cv2
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score, matthews_corrcoef,
                             precision_score, recall_score)

SEED=42
# Core conditions give a useful first study; set to 'all' for all 16 conditions.
CONDITIONS_MODE = 'core'  # 'core' or 'all'
N_BOOTSTRAPS = 300
DECODER_WORKERS = 4
REUSE_EXPERIMENT4_CACHE_IF_FOUND = True
# If there is no Experiment 6 model on disk, this will train the identical
# 150-estimator Trad-style XGBoost on the development set and save it locally.
RETRAIN_RAW_MODEL_IF_MISSING = True

MANUAL_FILE_PATHS = {
    'qr_codes_29.pickle': None,
    'qr_codes_29_labels.pickle': None,
    'qr_full_feature_dataset_v2.csv': None,
    'FINAL_10feature_XGB_for_deployment.pkl': None,
    'phase14B_final_robust_features.json': None,
    'trad_style_raw_pixel_xgb_grouped.pkl': None,   # optional if retrain enabled
    'sample_level_robustness_results.csv': None,    # optional (Experiment 4 cache)
    'esann_icann_psl_snapshot.txt': None,          # optional when tldextract is installed
}
ADDITIONAL_SEARCH_FOLDERS = [
    # r'D:\Your Jupyter Projects',
]
QR_DATA_WINDOWS = r'C:\Users\LENOVO\Desktop\FYP\QuishingDataset'
ALTERNATE_WINDOWS = r'C:\Users\LENOVO\FYP\QuishingDataset'
CWD = Path.cwd().resolve()
SCRIPT_DIR = Path(__file__).resolve().parent if '__file__' in globals() else None
OUT=CWD/'experiment7_matched_robustness'
OUT.mkdir(exist_ok=True, parents=True)



## 2. Locate files independently across Windows Desktop and Jupyter folders


In [ ]:
def _roots(seq):
    seen=set(); out=[]
    for x in seq:
        if x is None: continue
        try:
            p=Path(x).expanduser().resolve()
            if p.is_dir() and str(p).lower() not in seen:
                out.append(p); seen.add(str(p).lower())
        except (OSError, ValueError): pass
    return out

HOME=Path.home()
SEARCH_ROOTS=_roots([
    CWD,SCRIPT_DIR,CWD.parent,CWD/'FYP',CWD/'QuishingDataset',CWD/'archive',
    CWD/'experiment6_matched_trad_vs_lexical', CWD/'experiment4_holdout_corrected',
    CWD.parent/'experiment6_matched_trad_vs_lexical',
    CWD.parent/'experiment4_holdout_corrected',
    QR_DATA_WINDOWS,ALTERNATE_WINDOWS,
    HOME/'Desktop'/'FYP'/'QuishingDataset',HOME/'Desktop'/'FYP',
    HOME/'OneDrive'/'Desktop'/'FYP'/'QuishingDataset',
    HOME/'Documents',HOME/'Downloads',HOME/'Desktop',
    *ADDITIONAL_SEARCH_FOLDERS,
])
SKIP={'.git','.ipynb_checkpoints','__pycache__','venv','.venv','node_modules',
      'AppData','anaconda3','miniconda3','site-packages','experiment7_matched_robustness'}

def locate(filename, optional=False):
    override=MANUAL_FILE_PATHS.get(filename)
    if override:
        p=Path(override).expanduser()
        if not p.is_file(): raise FileNotFoundError(f'Invalid manual path for {filename}: {p}')
        return p.resolve()
    aliases=[filename]+(['qr_full_feature_dataset_v2.xls'] if filename.endswith('feature_dataset_v2.csv') else [])
    hits=[]; seen=set()
    def push(p):
        try:
            if p.is_file() and str(p.resolve()).lower() not in seen:
                hits.append(p.resolve());seen.add(str(p.resolve()).lower())
        except OSError: pass
    for root in SEARCH_ROOTS:
        for alias in aliases: push(root/alias)
        for sub in ['experiment6_matched_trad_vs_lexical', 'experiment4_holdout_corrected']:
            for alias in aliases:push(root/sub/alias)
    if not hits:
        for root in SEARCH_ROOTS:
            for folder, dirs, files in os.walk(root):
                depth=len(Path(folder).relative_to(root).parts)
                dirs[:]=[d for d in sorted(dirs) if d not in SKIP and not d.startswith('.') ] if depth < 3 else []
                for alias in aliases:
                    if alias in files: push(Path(folder)/alias)
    if not hits:
        if optional:return None
        raise FileNotFoundError(
            f'Cannot find {filename}. CWD={CWD}\n'
            f'Please set MANUAL_FILE_PATHS[{filename!r}] to its full Windows path.')
    if len(hits)>1:
        print(f'[NOTE] Multiple {filename} files found:')
        for p in hits[:5]:print('  ',p)
    if filename.endswith('.csv'):
        hits.sort(key=lambda p:p.suffix.lower()!='.csv')
    return hits[0]

REQUIRED=['qr_codes_29.pickle','qr_codes_29_labels.pickle','qr_full_feature_dataset_v2.csv',
          'FINAL_10feature_XGB_for_deployment.pkl','phase14B_final_robust_features.json']
PATHS={key:locate(key) for key in REQUIRED}
for key in ['trad_style_raw_pixel_xgb_grouped.pkl','sample_level_robustness_results.csv',
            'esann_icann_psl_snapshot.txt']:
    PATHS[key]=locate(key,optional=True)
print('Jupyter working directory:',CWD)
for k,v in PATHS.items():print(f'[FOUND] {k}: {v}' if v else f'[OPTIONAL NOT FOUND] {k}')
print('Output directory:', OUT)



## 3. Load aligned original QR data and freeze the same holdout as Experiment 6


In [ ]:
with open(PATHS['qr_codes_29.pickle'],'rb') as f:qr_codes=np.asarray(pickle.load(f))
with open(PATHS['qr_codes_29_labels.pickle'],'rb') as f:qr_labels=np.asarray(pickle.load(f)).astype(int)
df=pd.read_csv(PATHS['qr_full_feature_dataset_v2.csv'])
with open(PATHS['phase14B_final_robust_features.json'],encoding='utf8') as f:FEATURES=json.load(f)['features']
lex_model=joblib.load(PATHS['FINAL_10feature_XGB_for_deployment.pkl'])
assert qr_codes.shape==(9987,69,69),qr_codes.shape
assert qr_labels.shape==(9987,),qr_labels.shape
assert len(df)==9713,len(df)
assert df.original_index.is_unique and df.original_index.notna().all()
assert not df[['url','hostname','label']+FEATURES].isna().any().any()
orig=df.original_index.to_numpy(dtype=int)
y_all=df.label.to_numpy(dtype=int)
assert np.array_equal(qr_labels[orig],y_all),'Misaligned original_index to QR data'
assert np.isin(qr_codes,[0,1]).all(),'QR values expected to be 0=black,1=white'
assert len(FEATURES)==10
split=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
dev_idx,test_idx=next(split.split(df[FEATURES],y_all,groups=df.hostname.astype(str)))
assert (len(dev_idx),len(test_idx))==(7936,1777)
assert not (set(df.hostname.iloc[dev_idx]) & set(df.hostname.iloc[test_idx]))
held=df.iloc[test_idx].reset_index(drop=True)
y=held.label.to_numpy(dtype=int)
n=len(held)
orig_test=held.original_index.to_numpy(dtype=int)
host=held.hostname.astype(str).to_numpy()
X_lex=held[FEATURES]
lex_direct_prob=lex_model.predict_proba(X_lex)[:,1]
raw_features_original=qr_codes[orig_test].reshape(n,-1).astype(np.float32)
assert abs(roc_auc_score(y,lex_direct_prob)-0.950949)<0.0001, 'Unexpected lexical model / holdout mismatch'
print('Holdout:',n,'hostname overlap:',0,'labels:',held.label.value_counts().to_dict())
print('Frozen lexical AUC:',roc_auc_score(y,lex_direct_prob))



## 4. Load the exact Experiment 6 raw-pixel model or retrain the same configuration


In [ ]:
raw_path=PATHS['trad_style_raw_pixel_xgb_grouped.pkl']
if raw_path is None:
    if not RETRAIN_RAW_MODEL_IF_MISSING:
        raise FileNotFoundError('Experiment 6 raw model not found. Run Exp6 or set manual path.')
    print('[MODEL] Exp6 model missing; retraining on original development rows only.')
    raw_model=XGBClassifier(learning_rate=.2,n_estimators=150,eval_metric='logloss',
                            random_state=SEED,n_jobs=4)
    raw_model.fit(qr_codes[orig[dev_idx]].reshape(len(dev_idx),4761).astype(np.float32),y_all[dev_idx])
    raw_path=OUT/'trad_style_raw_pixel_xgb_grouped_RECREATED.pkl'
    joblib.dump(raw_model,raw_path)
else:
    print('[MODEL] Loading frozen Exp6 raw-pixel model:',raw_path)
    raw_model=joblib.load(raw_path)
raw_clean_prob=raw_model.predict_proba(raw_features_original)[:,1]
assert abs(roc_auc_score(y,raw_clean_prob)-0.88218)<0.002, (
    'Loaded raw-pixel model does not reproduce Exp6 AUC=0.88218! STOP.')
print('Trad-style raw-pixel clean AUC:',roc_auc_score(y,raw_clean_prob))



## 5. Use EXACT distortion transforms from the corrected Experiment 4


In [ ]:
cv2.setNumThreads(1)
MODULE_SCALE=10
BORDER_MODULES=4
BASE_SIDE=(69+2*BORDER_MODULES)*MODULE_SCALE  # 770 pixels

def render_qr(matrix):
    base=(np.asarray(matrix,dtype=np.uint8)*255).astype(np.uint8)
    base=cv2.copyMakeBorder(base,4,4,4,4,cv2.BORDER_CONSTANT,value=255)
    return cv2.resize(base,None,fx=10,fy=10,interpolation=cv2.INTER_NEAREST)

def apply_blur(img,sigma):
    k=max(3,int(2*round(3*sigma)+1))
    if k%2==0:k+=1
    return cv2.GaussianBlur(img,(k,k),sigmaX=sigma,sigmaY=sigma)

def rotate_bound_white(img,angle):
    h,w=img.shape[:2];center=(w/2,h/2)
    M=cv2.getRotationMatrix2D(center,angle,1.0)
    cos,sin=abs(M[0,0]),abs(M[0,1])
    nw=int(h*sin+w*cos);nh=int(h*cos+w*sin)
    M[0,2]+=nw/2-center[0];M[1,2]+=nh/2-center[1]
    return cv2.warpAffine(img,M,(nw,nh),flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT,borderValue=255)

def apply_jpeg(img,quality):
    ok,b=cv2.imencode('.jpg',img,[int(cv2.IMWRITE_JPEG_QUALITY),int(quality)])
    if not ok:raise RuntimeError('JPEG encoding failed')
    return cv2.imdecode(b,cv2.IMREAD_GRAYSCALE)

def apply_scale(img,scale):
    h,w=img.shape[:2]
    small=cv2.resize(img,(max(8,int(round(w*scale))),max(8,int(round(h*scale)))),
                     interpolation=cv2.INTER_AREA)
    return cv2.resize(small,(w,h),interpolation=cv2.INTER_LINEAR)

def distort(img,kind,severity):
    if kind=='clean':return img.copy()
    if kind=='blur':return apply_blur(img,float(severity))
    if kind=='rotation':return rotate_bound_white(img,float(severity))
    if kind=='jpeg':return apply_jpeg(img,int(severity))
    if kind=='scale':return apply_scale(img,float(severity))
    raise ValueError((kind,severity))

ALL_CONDITIONS=[('clean',0),('blur',.5),('blur',1.0),('blur',1.5),
                ('rotation',-15),('rotation',-10),('rotation',-5),
                ('rotation',5),('rotation',10),('rotation',15),
                ('jpeg',80),('jpeg',50),('jpeg',30),
                ('scale',.75),('scale',.5),('scale',.25)]
CORE_CONDITIONS=[('clean',0),('blur',1.5),('rotation',-10),('rotation',10),
                 ('rotation',15),('jpeg',30),('scale',.25)]
CONDITIONS=ALL_CONDITIONS if CONDITIONS_MODE=='all' else CORE_CONDITIONS

def cond_name(kind,sev):
    if kind=='clean':return 'Clean'
    if kind=='blur':return f'Blur_sigma_{sev}'
    if kind=='rotation':return f'Rotation_{"plus" if float(sev)>0 else "minus"}_{abs(float(sev))}deg'
    if kind=='jpeg':return f'JPEG_Q{sev}'
    if kind=='scale':return f'Scale_{sev}'
    return f'{kind}_{sev}'

# IMPORTANT raw-pixel frontend:
# The same distorted raster passed to the QR decoder is first resized to the
# original 770x770 canvas (if rotation expanded its bounding box), then the
# *nominal* 4-module quiet zone is cropped, 690x690 -> 69x69 sampled and
# binarized. This is deliberately fixed, NOT a learned detector and NOT
# geometric rectification, so rotation can strongly perturb the raw baseline.
# Clean condition MUST exactly reconstruct the original 69x69 module grid.

def distorted_image_to_raw_pixels(image):
    if image.shape[:2]!=(BASE_SIDE,BASE_SIDE):
        image=cv2.resize(image,(BASE_SIDE,BASE_SIDE),interpolation=cv2.INTER_LINEAR)
    margin=MODULE_SCALE*BORDER_MODULES
    crop=image[margin:BASE_SIDE-margin,margin:BASE_SIDE-margin]
    small=cv2.resize(crop,(69,69),interpolation=cv2.INTER_AREA)
    return (small>=128).astype(np.float32).reshape(4761)

for j in [0,n//2,n-1]:
    grid=distorted_image_to_raw_pixels(render_qr(qr_codes[orig_test[j]]))
    assert np.array_equal(grid,raw_features_original[j]),'Clean raw pixels do not match training pixels'
print('Clean reconstruction: native QR -> 770px render -> 69x69 raw pixels = EXACT')



## 6. Load verified Exp4 decoded-URL cache when present; otherwise decode live

Using an Exp4 cache is methodologically equivalent only after we match every
condition, original_index and label against the current frozen holdout and
verify clean predictions. A cached decoder score is never substituted for a
different QR transform or sample set.


In [ ]:
def normalize_url(url):
    raw=str(url).strip(); inp=raw if raw.lower().startswith(('http://','https://')) else 'http://'+raw
    parsed=urlparse(inp)
    hostname=parsed.hostname.lower() if parsed.hostname else ''
    if hostname.startswith('www.'): hostname=hostname[4:]
    try:port=parsed.port
    except ValueError:port=None
    netloc=hostname+(':'+str(port) if port is not None else '')
    return urlunparse((parsed.scheme.lower(),netloc,parsed.path,'',parsed.query,''))

CACHE=None
cache_file=PATHS['sample_level_robustness_results.csv']
if REUSE_EXPERIMENT4_CACHE_IF_FOUND and cache_file:
    temp=pd.read_csv(cache_file)
    cache_ready=True
    for kind,sev in CONDITIONS:
        subset=temp.loc[temp.condition==cond_name(kind,sev)].copy()
        if len(subset)!=n:
            cache_ready=False;break
        subset=subset.sort_values('sample_row').reset_index(drop=True)
        if not (np.array_equal(subset.original_index.to_numpy(dtype=int),orig_test) and
                np.array_equal(subset.true_label.to_numpy(dtype=int),y) and
                np.array_equal(subset.true_url.astype(str),held.url.astype(str))):
            cache_ready=False;break
    if cache_ready:
        clean=temp.loc[temp.condition=='Clean'].sort_values('sample_row').reset_index(drop=True)
        clean_probs=clean.predicted_probability.to_numpy(dtype=float)
        if not (len(clean)==n and clean.correct_decode.astype(bool).all() and
                np.max(np.abs(clean_probs-lex_direct_prob))<1e-9):
            cache_ready=False
    if cache_ready:
        CACHE=temp
        print('[CACHE] Using verified Exp4 per-image decode / lexical inference results:',cache_file)
    else:
        print('[CACHE] Found old Exp4 data but audit failed; recomputing live instead.')
else:
    print('[CACHE] No Exp4 result cache detected; decoding QR images live.')



## 7. Exact deployment extractor for live decoding (only needed without cache)
Public suffix handling MUST match the frozen training feature table.


In [ ]:
SUSPICIOUS_KEYWORDS=['login','verify','verification','account','secure','update',
                     'password','signin','confirm','payment','bank','billing','support']
DOMAIN_EXTRACTOR=None
PSL_EXACT=PSL_WILD=PSL_EXCEPT=None
if CACHE is None:
    try:
        import tldextract
        DOMAIN_EXTRACTOR=tldextract.TLDExtract(suffix_list_urls=None)
        print('[DEPLOYMENT] Using offline tldextract.')
    except ImportError:
        psl_path=PATHS['esann_icann_psl_snapshot.txt']
        # Accept a PSL snapshot stored in the original complete correction ZIP.
        psl=None
        if psl_path is not None:
            psl=Path(psl_path).read_text(encoding='utf8').splitlines()
        else:
            for root in SEARCH_ROOTS:
                zp=root/'ESANN_Corrected_Experiments_Full_Package.zip'
                if zp.is_file():
                    with zipfile.ZipFile(zp) as zf:
                        try:
                            psl=zf.read('esann_icann_psl_snapshot.txt').decode('utf8').splitlines()
                            print('[DEPLOYMENT] PSL loaded from correction ZIP:',zp)
                            break
                        except KeyError:pass
        if psl is None:
            raise ImportError('Cannot find tldextract or esann_icann_psl_snapshot.txt.\n'
                              'Run `pip install tldextract` or put the PSL snapshot beside the notebook.\n'
                              'Alternatively, provide the validated Experiment 4 sample_level_robustness_results.csv.')
        psl=[s.strip() for s in psl if s.strip()]
        PSL_EXACT={s for s in psl if not s.startswith(('*.','!'))}
        PSL_WILD={s[2:] for s in psl if s.startswith('*.')}
        PSL_EXCEPT={s[1:] for s in psl if s.startswith('!')}
        print('[DEPLOYMENT] Offline fixed PSL fallback active.')

def get_registered_domain(hostname):
    hostname=str(hostname).lower().strip('.')
    if not hostname:return ''
    if hostname=='fed.us' or hostname.endswith('.fed.us'):return 'fed.us'
    if DOMAIN_EXTRACTOR is not None:
        ext=DOMAIN_EXTRACTOR(hostname)
        if not ext.domain:return hostname
        return ext.domain+'.'+ext.suffix if ext.suffix else ext.domain
    labels=hostname.split('.'); public_suffix_size=1
    for i in range(len(labels)):
        candidate='.'.join(labels[i:])
        if candidate in PSL_EXCEPT:
            public_suffix_size=len(labels)-i-1;break
        if candidate in PSL_EXACT:
            public_suffix_size=max(public_suffix_size,len(labels)-i)
        if i<len(labels)-1 and '.'.join(labels[i+1:]) in PSL_WILD:
            public_suffix_size=max(public_suffix_size,len(labels)-i)
    return '.'.join(labels[-(public_suffix_size+1):]) if len(labels)>public_suffix_size else hostname

def deployment_features(url):
    raw=str(url).strip()
    inp=raw if re.match(r'^[A-Za-z][A-Za-z0-9+.\-]*://',raw) else 'http://'+raw
    try:raw_hostname=(urlsplit(inp).hostname or '').lower()
    except Exception:raw_hostname=''
    norm=normalize_url(raw)
    try:p=urlsplit(norm);path=p.path or '';query=p.query or ''
    except Exception:path=query=''
    n=len(raw); domain=get_registered_domain(raw_hostname)
    return {'num_subdomains':max(len(raw_hostname.strip('.').split('.'))-len(domain.strip('.').split('.')),0) if raw_hostname else 0,
            'suspicious_keyword_count':sum(w in raw.lower() for w in SUSPICIOUS_KEYWORDS),
            'num_digits':sum(c.isdigit() for c in raw),
            'path_length':len(path), 'hostname_length':len(raw_hostname),
            'url_length':n, 'num_dots':raw.count('.'),
            'special_char_ratio':sum(not c.isalnum() for c in raw)/n if n else 0.,
            'num_hyphens':raw.count('-'), 'query_length':len(query)}

if CACHE is None:
    # Sanity-audit against ALL 9713 model training-table feature rows.
    # Ensures no silent change to lexical parser between experiments.
    test_features=pd.DataFrame([deployment_features(u) for u in df.url.astype(str)])[FEATURES]
    delta=np.abs(test_features.to_numpy(dtype=float)-df[FEATURES].to_numpy(dtype=float))
    nbad=int(np.count_nonzero(np.max(delta,axis=1)>1e-10))
    print('Deployment consistency mismatching rows:',nbad,'of',len(df))
    if nbad:
        mismatch_rows=np.flatnonzero(np.max(delta,axis=1)>1e-10)[:8]
        print('First mismatch rows:',mismatch_rows)
        raise RuntimeError('Deployment extractor not identical to frozen training features! Stop.')



## 8. Score the SAME distorted images through both pipelines

Images are generated once per sample-condition. Raw-pixel scoring is batched.
Decoder outputs are either recomputed on these images or loaded from a strictly
sample-matched previous Experiment 4 run.


In [ ]:
_tls=threading.local()

def _decode_worker(image):
    d=getattr(_tls,'detector',None)
    if d is None:
        d=cv2.QRCodeDetector();_tls.detector=d
    txt,_,_=d.detectAndDecode(image)
    return str(txt).strip() if txt else ''

rows=[]
for ci,(kind,sev) in enumerate(CONDITIONS):
    t0=time.perf_counter(); cname=cond_name(kind,sev)
    raw_values=np.empty((n,4761),dtype=np.float32)
    decoded=[]; cached=None
    if CACHE is not None:
        cached=CACHE.loc[CACHE.condition==cname].sort_values('sample_row').reset_index(drop=True)
    if cached is None:executor=ThreadPoolExecutor(max_workers=DECODER_WORKERS)
    try:
        for i in range(n):
            image=distort(render_qr(qr_codes[orig_test[i]]),kind,sev)
            raw_values[i]=distorted_image_to_raw_pixels(image)
            if cached is None:
                decoded.append(executor.submit(_decode_worker,image))
        if cached is None:
            decoded=[f.result() for f in decoded]
        else:
            decoded=[x if isinstance(x,str) and x.strip() else '' for x in cached.decoded_url]
    finally:
        if cached is None:executor.shutdown(wait=True)
    raw_probs=raw_model.predict_proba(raw_values)[:,1]
    if cname=='Clean':
        assert np.array_equal(raw_values,raw_features_original), (
            'Clean transformed raw input differs from Experiment 6 original features')
        assert np.max(np.abs(raw_probs-raw_clean_prob))<1e-10, (
            'Clean raw-model predictions differ from Experiment 6!')
    decoded_out=np.array([bool(d) for d in decoded],dtype=bool)
    correct_payload=np.array([bool(d) and normalize_url(d)==normalize_url(u)
                              for d,u in zip(decoded,held.url.astype(str))],dtype=bool)
    if cached is not None:
        assert np.array_equal(correct_payload,cached.correct_decode.to_numpy(dtype=bool)),(
            'Cached decoding disagrees with current normalization')
        lex_probs=cached.predicted_probability.to_numpy(dtype=float)
        lex_probs[~decoded_out]=np.nan
    else:
        lex_probs=np.full(n,np.nan)
        idx=np.flatnonzero(decoded_out)
        if len(idx):
            x_dec=pd.DataFrame([deployment_features(decoded[i]) for i in idx])[FEATURES]
            lex_probs[idx]=lex_model.predict_proba(x_dec)[:,1]
    lex_pred=np.where(np.isfinite(lex_probs),lex_probs>=0.5,False).astype(int)
    raw_pred=(raw_probs>=0.5).astype(int)
    if cname=='Clean':
        assert correct_payload.all(),'Unexpected Clean decoding failure!'
        assert np.nanmax(np.abs(lex_probs-lex_direct_prob))<1e-10,(
            'Clean decoded lexical probabilities differ from frozen lexical input')
    condition_data=pd.DataFrame({
        'condition':cname,'kind':kind,'severity':sev,'sample_row':np.arange(n),
        'original_index':orig_test,'hostname':host,'true_label':y,
        'decode_nonempty':decoded_out,'decode_correct_payload':correct_payload,
        'raw_probability':raw_probs,'raw_prediction':raw_pred,
        'raw_correct':raw_pred==y,'lexical_probability':lex_probs,
        'lexical_prediction':np.where(decoded_out,lex_pred,np.nan),
        'lexical_correct_given_correct_decode':correct_payload & (lex_pred==y),
        'lexical_strict_correct':correct_payload & (lex_pred==y),
        'raw_correct_when_lexical_decode_fails':(~correct_payload)&(raw_pred==y),
        'raw_pixel_change_fraction':np.mean(raw_values!=raw_features_original,axis=1),
    })
    rows.append(condition_data)
    print(f'{ci+1:2}/{len(CONDITIONS)} {cname:<23} '
          f'Decode={correct_payload.mean():.3%} '
          f'Raw-AUC={roc_auc_score(y,raw_probs):.4f} '
          f'Raw-E2E={condition_data.raw_correct.mean():.3%} '
          f'Lex-E2E={condition_data.lexical_strict_correct.mean():.3%} '
          f'elapsed={time.perf_counter()-t0:.1f}s',flush=True)
    # Save after every condition for reproducible partial progress.
    condition_data.to_csv(OUT/f'sample_{cname}.csv',index=False)
all_rows=pd.concat(rows,ignore_index=True)
all_rows.to_csv(OUT/'matched_sample_level_results.csv',index=False)



## 9. Fair metric definitions: full-population vs conditional subsets
Raw always produces a score. The decoding-based system may abstain.
Never compare raw full-set ROC--AUC to lexical *conditional* ROC--AUC as
though they use the same denominator. Use a common correct-decode subset.


In [ ]:
def auc_safe(yy,prob):
    if len(np.unique(yy))<2:return float('nan')
    return roc_auc_score(yy,prob)

def binary_metrics(yy,score):
    pred=(score>=0.5).astype(int)
    return dict(auc=auc_safe(yy,score), f1=f1_score(yy,pred,zero_division=0),
                mcc=matthews_corrcoef(yy,pred), accuracy=accuracy_score(yy,pred),
                precision=precision_score(yy,pred,zero_division=0),
                recall=recall_score(yy,pred,zero_division=0))

summary=[]
for (kind,sev),r in zip(CONDITIONS,rows):
    good=r.decode_correct_payload.to_numpy(dtype=bool)
    raw=r.raw_probability.to_numpy();lex=r.lexical_probability.to_numpy()
    dec_fail=~good
    full_raw=binary_metrics(y,raw)
    cond_raw=binary_metrics(y[good],raw[good]) if good.sum()>0 else dict.fromkeys(binary_metrics(y,raw).keys(),np.nan)
    cond_lex=binary_metrics(y[good],lex[good]) if good.sum()>0 else dict.fromkeys(binary_metrics(y,raw).keys(),np.nan)
    phishing=(y==1)
    raw_pred=(raw>=0.5)
    row=dict(condition=cond_name(kind,sev),kind=kind,severity=sev,n=n,
             raw_coverage=1.0,lexical_nonempty_decode_rate=r.decode_nonempty.mean(),
             lexical_correct_payload_rate=good.mean(),
             raw_full_auc=full_raw['auc'],raw_full_f1=full_raw['f1'],
             raw_predicted_phishing_rate=float(raw_pred.mean()),
             raw_predicted_phishing_rate_given_decode_failure=float(raw_pred[dec_fail].mean()) if np.any(dec_fail) else np.nan,
             raw_full_mcc=full_raw['mcc'],raw_full_accuracy=full_raw['accuracy'],
             raw_common_subset_auc=cond_raw['auc'],lex_common_subset_auc=cond_lex['auc'],
             raw_common_subset_f1=cond_raw['f1'],lex_common_subset_f1=cond_lex['f1'],
             raw_end_to_end_correct_rate=(raw_pred==y).mean(),
             lexical_strict_end_to_end_correct_rate=r.lexical_strict_correct.mean(),
             lexical_abstention_or_payload_failure=int(dec_fail.sum()),
             raw_correct_among_decode_failures=int(((raw_pred==y)&dec_fail).sum()),
             raw_correct_rate_among_decode_failures=(raw_pred[dec_fail]==y[dec_fail]).mean() if dec_fail.sum() else np.nan,
             decode_failure_phishing_count=int((dec_fail&phishing).sum()),
             raw_phishing_recall_among_decode_failures=(raw_pred[dec_fail&phishing]==1).mean() if np.any(dec_fail&phishing) else np.nan,
             raw_pixel_change_fraction=r.raw_pixel_change_fraction.mean())
    summary.append(row)
summary_df=pd.DataFrame(summary)
cols=['condition','lexical_correct_payload_rate','raw_full_auc','raw_full_f1',
      'raw_end_to_end_correct_rate','lexical_strict_end_to_end_correct_rate',
      'raw_correct_among_decode_failures','lexical_abstention_or_payload_failure']
print('\n=== Matched robustness comparison ===')
print(summary_df[cols].round(5).to_string(index=False))
summary_df.to_csv(OUT/'matched_robustness_summary.csv',index=False)



## 10. Paired hostname bootstrap for the full-population end-to-end difference
We compare two binary success indicators on exactly the same sample IDs.
The conclusion is not about AUC when one pipeline abstains.


In [ ]:
unique_hosts,inverse=np.unique(host,return_inverse=True)
# Efficient, EXACT hostname-cluster resampling: for every drawn hostname,
# include all of its rows. Aggregate cluster sizes / success counts instead
# of concatenating thousands of row-index arrays on every bootstrap iteration.
cluster_n=np.bincount(inverse, minlength=len(unique_hosts)).astype(np.float64)
rng=np.random.default_rng(SEED)
cluster_draws=rng.integers(0,len(unique_hosts),size=(N_BOOTSTRAPS,len(unique_hosts)))
boot_denominator=cluster_n[cluster_draws].sum(axis=1)
bs=[]
for k,r in enumerate(rows):
    raw_ok=r.raw_correct.to_numpy(dtype=bool)
    lex_ok=r.lexical_strict_correct.to_numpy(dtype=bool)
    cluster_delta=np.bincount(inverse,weights=(raw_ok.astype(float)-lex_ok.astype(float)),
                              minlength=len(unique_hosts))
    diffs=cluster_delta[cluster_draws].sum(axis=1)/boot_denominator
    lower,upper=np.quantile(diffs,[0.025,0.975])
    bs.append(dict(condition=cond_name(*CONDITIONS[k]),
                   delta_raw_minus_lex_e2e=float(raw_ok.mean()-lex_ok.mean()),
                   ci95_lower=float(lower),ci95_upper=float(upper),
                   bootstrap_replicates=N_BOOTSTRAPS,cluster='hostname'))
bootstrap_summary=pd.DataFrame(bs)
bootstrap_summary.to_csv(OUT/'paired_hostname_bootstrap_e2e.csv',index=False)
print('\n=== Paired bootstrap: raw minus lexical, all 1,777 rows ===')
print(bootstrap_summary.round(5).to_string(index=False))



## 11. Visual summary and handoff


In [ ]:
fig,ax=plt.subplots(figsize=(11,4.5))
x=np.arange(len(summary_df))
ax.plot(x,summary_df['raw_end_to_end_correct_rate'].to_numpy(),marker='o',label='Raw pixel: full-sample correct')
ax.plot(x,summary_df['lexical_strict_end_to_end_correct_rate'].to_numpy(),marker='s',label='Decode + lexical: strict full-sample correct')
ax.plot(x,summary_df['lexical_correct_payload_rate'].to_numpy(),marker='^',ls='--',label='Lexical: correct decode rate')
ax.set_xticks(x,summary_df.condition,rotation=36,ha='right')
ax.set_ylim(0,1.03);ax.set_ylabel('Fraction of identical 1,777-sample holdout')
ax.set_title('Experiment 7: Matched robustness, same images and labels')
ax.grid(axis='y',alpha=.25);ax.legend(loc='best',fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'exp7_matched_robustness.png',dpi=160,bbox_inches='tight')
plt.show()

metadata=dict(seed=SEED,conditions_mode=CONDITIONS_MODE,
              conditions=[cond_name(a,b) for a,b in CONDITIONS],
              n_holdout=n,hostname_overlap=0,used_experiment4_cached_outputs=CACHE is not None,
              source_files={k:str(v) if v else None for k,v in PATHS.items()},
              raw_model_path=str(raw_path),
              raster_preprocessing='resize entire distorted image to 770x770 (including expanded rotation canvas), crop central 40px quiet zone, 69x69 INTER_AREA, threshold >=128',
              same_image=True, raw_model_augmented_for_distortions=False,
              strict_lexical_success='correct decoded payload AND correct binary prediction',
              raw_success='correct classification on all images; coverage=1',
              limitation='Center-crop raw-pixel frontend lacks rotation alignment and is not optimized for distortions.',
              package_versions={'opencv':cv2.__version__,'pandas':pd.__version__})
(OUT/'exp7_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf8')
print('\nAll results saved to',OUT.resolve())
print('FILES: matched_robustness_summary.csv, paired_hostname_bootstrap_e2e.csv, exp7_matched_robustness.png')
print('Important: compare FULL end-to-end correct rates on the same denominator; common-subset AUC separately.')

print('CAVEAT: A raw-pixel frontend that predicts only one class after rotation is degenerate; use the predicted-phishing-rate field before claiming useful fallback detection.')


In [ ]:
check_cols = [
    "condition",
    "raw_pixel_change_fraction",
    "raw_predicted_phishing_rate",
    "raw_full_mcc",
    "raw_correct_rate_among_decode_failures"
]

print(summary_df[check_cols].round(4).to_string(index=False))

In [ ]:

# EXP7 Additional Diagnostic:
# Oracle rotation correction for Trad-style raw-pixel model

from sklearn.metrics import (
    roc_auc_score, f1_score,
    matthews_corrcoef, accuracy_score
)

def oracle_restore_rotation(image, angle):
    h, w = image.shape[:2]
    center = (w / 2, h / 2)

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    c, s = abs(M[0, 0]), abs(M[0, 1])

    new_w = int(h * s + w * c)
    new_h = int(h * c + w * s)

    M[0, 2] += new_w / 2 - center[0]
    M[1, 2] += new_h / 2 - center[1]

    rotated = cv2.warpAffine(
        image, M, (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255
    )

    restored = cv2.warpAffine(
        rotated,
        cv2.invertAffineTransform(M),
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255
    )

    return restored


results = []

for angle in [-10, 10, 15]:
    X_restored = np.empty_like(raw_features_original)

    for i, original_id in enumerate(orig_test):
        image = render_qr(qr_codes[original_id])

        restored = oracle_restore_rotation(image, angle)

        X_restored[i] = distorted_image_to_raw_pixels(
            restored
        )

    probabilities = raw_model.predict_proba(
        X_restored
    )[:, 1]

    predictions = (probabilities >= 0.5).astype(int)

    pixel_change = np.mean(
        X_restored != raw_features_original
    )

    results.append({
        "angle": angle,
        "roc_auc": roc_auc_score(y, probabilities),
        "f1": f1_score(y, predictions),
        "mcc": matthews_corrcoef(y, predictions),
        "accuracy": accuracy_score(y, predictions),
        "pixel_change_fraction": pixel_change,
        "predicted_phishing_rate": predictions.mean()
    })

oracle_results = pd.DataFrame(results)

print("\n=== Oracle Rotation Correction ===")
print(oracle_results.round(5).to_string(index=False))

oracle_results.to_csv(
    OUT / "oracle_rotation_correction.csv",
    index=False
)

print("\nResults saved successfully.")


In [ ]:

print(
    summary_df.columns.tolist()
)

print(
    summary_df.loc[
        summary_df["condition"] == "Rotation_plus_10.0deg"
    ].T.to_string()
)


In [ ]:

from pathlib import Path
import pandas as pd

# Find Exp7 CSV files in the current project folder
files = [
    p for p in Path(".").rglob("*.csv")
    if any(k in str(p).lower()
           for k in ["exp7", "robust", "rotation"])
]

print("Found files:")
for p in files:
    print(p)

# If no files appear, check the folder used
# by the notebook when saving its results.


In [ ]:
import pandas as pd
from pathlib import Path

# This is the actual file found in your screenshot
FILE = (
    Path("experiment7_matched_robustness")
    / "sample_Rotation_plus_10.0deg.csv"
)

df = pd.read_csv(FILE)

print("文件是否存在：", FILE.exists())
print("总行数：", len(df))

print("\n所有列名：")
print(df.columns.tolist())

print("\n前5行数据：")
print(df.head().to_string(index=False))


In [ ]:
import pandas as pd
from pathlib import Path

summary_file = (
    Path("experiment7_matched_robustness")
    / "matched_robustness_summary.csv"
)

summary = pd.read_csv(summary_file)

print("汇总文件列名：")
print(summary.columns.tolist())

print("\n完整汇总结果：")
print(summary.to_string(index=False))


In [ ]:

import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

# Your actual Exp7 sample-level CSV
file = (
    Path("experiment7_matched_robustness")
    / "sample_Rotation_plus_10.0deg.csv"
)

df = pd.read_csv(file)

# Correct decoding: recovered payload matches
# the original URL
decode_ok = (
    df["decode_correct_payload"]
    .astype(str).str.lower().eq("true")
)

# Full-sample correct decision
strict_correct = (
    df["lexical_strict_correct"]
    .astype(str).str.lower().eq("true")
)

# Independently verify strict correctness
expected_correct = (
    decode_ok &
    df["lexical_prediction"].eq(df["true_label"])
)

assert len(df) == 1777
assert strict_correct.equals(expected_correct), (
    "Saved strict correctness does not match "
    "decoded payload and predicted label."
)

# Recompute conditional AUC from probabilities
valid = df.loc[decode_ok].copy()

assert valid["lexical_probability"].notna().all()
assert valid["true_label"].nunique() == 2

auc = roc_auc_score(
    valid["true_label"].astype(int),
    valid["lexical_probability"].astype(float)
)

print("Total samples:", len(df))
print("Correctly decoded:", decode_ok.sum())
print("Decode rate:", f"{decode_ok.mean()*100:.2f}%")
print("Correct lexical decisions:", strict_correct.sum())
print("Full-sample lexical correctness:",
      f"{strict_correct.mean()*100:.2f}%")
print("Conditional lexical AUC:", f"{auc:.4f}")

# Verify the numbers reported in the paper
assert round(decode_ok.mean()*100, 2) == 68.20
assert round(strict_correct.mean()*100, 2) == 60.16
assert round(auc, 4) == 0.9478

print("\nPASS: All three paper values reproduced.")
